In [2]:
import subprocess
import pandas as pd
from multiprocessing import Pool
import time, ipaddress, json, csv, ipaddress, random, datetime, sys, os
from collections import defaultdict
from  backgroundFunctions import *
from create_trees import TreeNode
from load_trees import load_tree_from_json
from collections import defaultdict, deque
import random 
import subprocess
import shutil
import os
import numpy as np


In [3]:
profileDir= '/mnt/md0/jaber/bgtraffic/ctp_50_cluster_latest/' # The location of output profiles directory
subprocess.call(['mkdir', '-p' ,profileDir])
profilesInfoFile =  profileDir[:-1] + '.json'
pcapDir =  '/mnt/md0/jaber/bgtraffic/timestampedPackets-outgoing-1/' # Don't change this 

# Loading the trees

In [4]:
# Load all 15 tree files into a list and create profiles (the index corresponds to tree number)
rootList = []
for i in range(15):
    tree_file = f'{tree_loc}/tree_nodes_{i}.json'
    rootNode = load_tree_from_json(tree_file)
    rootList.append(rootNode)

# Selecting the background profiles based on disired criteria 

In [5]:
# profiles --> fill it with a list of desired profiles (based on clusters and the list I gave you) 
ctps = pd.read_csv('/home/jaber/netReplica/netburst_latest_50_clusters_samples_with_timeseries.csv')


In [6]:
# For each row in the comma-separated ctps dataframe, for each row:
#   - get the "ip" and "index" columns
#   - get the root node from rootList using the "index"
#   - find the node in the tree corresponding to the "ip" subnet,
#   - get its children (leaf nodes under that subnet)
#   - store a profile dictionary with key as profile name (maybe f"cluster{cluster}_profile{id}"), and value as [ip, index, cluster, list_of_users] (you'll only append the users in the next cell)
#   - create 'profiles' dictionary as output as described for use by the next cell

profiles = dict()
for idx, row in ctps.iterrows():

    ip_subnet = str(row['ip'])
    tree_index = int(row['index'])
    cluster = int(row['cluster'])
    rootNode = rootList[tree_index]

    # profile name: e.g. 'cluster41_profile0'
    profile_name = f"cluster{cluster}_tree{tree_index}_profile{idx}"

    
    # Store: [ip, index, cluster] as initial info, append users in next cell
    profiles[profile_name] = [ip_subnet, tree_index, cluster]


In [7]:

def get_all_profiles_tree_based1(rootList, profiles, outputFile):

    for prof , info in profiles.items():
        try:

            print(info)
            tree_index = info[1]
            users = getUsersOfProfile(rootList[tree_index] , info[0])
            # append the users to the profile
            profiles[prof] = [  info[0],  [0, 0, 0, tree_index *2, (tree_index + 1)*2 ], users]
        except:
            print('error')
            print(f'The profile {prof} is not valid')
            continue
    # store the profiles in a json file
    with open(outputFile, 'w') as f:
        json.dump(profiles, f)

In [ ]:
get_all_profiles_tree_based1(rootList, profiles, profilesInfoFile)

In [9]:
len(profiles)



5000

# Processing the uplink CTP

## Aggregate user traffics for each profile 

In [ ]:
# Merging pcap files of all users belonging to the same profile into one pcap file
MergePcapsOfProfiles1(profilesInfoFile, pcapDir, profileDir)

## Fix packet ordering 

In [ ]:
# Reordering the packets in the merged pcap files based on the timestamp, in case they are not in correct order
reorder_pcap_files(profileDir)

## Padding the pcaps with dummy payload

In [ ]:
# Padding the pcap files to match the correct value of the ip length field
feed_pcap_files_for_padding(profileDir) ## There are several other implementtaions that can help


## Trimming to desired bandwidth

In [ ]:
# Trimming the pcap files to a desired number of packets and choosing the right destination for that
# profileDir= '/mnt/md0/jaber/bgtraffic/ctp_100_cluster/' # The location of output profiles directory
trimDir = profileDir[:-1] + '_6M/'  # The naming should be based on trimming size
# trimDir = profileDir[:-1] + '_trim/'
subprocess.call(['mkdir', '-p' ,trimDir])


feed_pcap_files_for_trimming(profileDir,trimDir, 75000) # The third argument is the desired throughput (it is in Byte, you should convert Mbps to bytes)

# Processing the downlink CTP

In [15]:
# Run the code either on upstream server (server 2) or do it here and send the CTPs there later.
# everything would be the same, just preprocessing the files and then moving them to server 2
pcapDirIncoming =  '/mnt/md0/jaber/bgtraffic/timestampedPackets-incoming-1/'
profileDirIncoming = profileDir[:-1] + '_incoming/'
print(profileDirIncoming)
subprocess.call(['mkdir', '-p' ,profileDirIncoming])


/mnt/md0/jaber/bgtraffic/ctp_50_cluster_latest_incoming/


0

In [ ]:
MergePcapsOfProfiles1(profilesInfoFile, pcapDirIncoming , profileDirIncoming )
reorder_pcap_files(profileDirIncoming )


In [ ]:
feed_pcap_files_for_padding(profileDirIncoming )  

In [8]:
trimDirIncoming = profileDirIncoming[:-1] + '_10M/'
subprocess.call(['mkdir', '-p' ,trimDirIncoming])
print(trimDirIncoming)

/mnt/md0/jaber/bgtraffic/ctp_100_cluster_incoming_10M/


In [ ]:
feed_pcap_files_for_trimming(profileDirIncoming, trimDirIncoming, 125000)